# Projeto Integrador IV — Ciência de Dados aplicado a Situações de Mercado
### Etapa 1 — Análise Exploratória e Visualização de Dados

**Curso:** Banco de Dados — SENAC EAD
**Disciplina:** Mineração de Dados

**Integrantes do grupo:**
- ARNO WENDT FILHO
- DAVID CALAZANS TAVARES
- RAFAELLA SILVA
- RHAYANE BERNARDINO SOUZA LEAO

> ⚠️ Antes de entregar: confira se todos os integrantes acima foram adicionados como colaboradores do repositório no GitHub (é item de rubrica).


## 1. Fonte de dados escolhida

**Fonte:** *Brazilian E-Commerce Public Dataset by Olist* — Kaggle
Link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

**O que é:** mais de 100 mil pedidos reais e anonimizados feitos entre 2016 e 2018 na **Olist
Store**, um marketplace brasileiro que conecta pequenos e médios vendedores a grandes
marketplaces (Mercado Livre, Americanas, etc.). O conjunto é dividido em várias tabelas
relacionadas: pedidos, itens do pedido, clientes, vendedores, produtos, pagamentos e avaliações.

**Por que essa fonte se encaixa em "situação de mercado":** é um cenário real de e-commerce
brasileiro em que a gestão precisa decidir, por exemplo, em quais categorias de produto investir,
para quais estados focar logística e marketing, como está o prazo de entrega (fator crítico de
satisfação) e quais formas de pagamento priorizar no checkout — decisões de negócio apoiadas por
dados.

## 2. Escopo da exploração

- **Recorte:** pedidos com status `delivered` (efetivamente entregues — o que representa vendas
  concretizadas), usando as tabelas `orders`, `order_items`, `customers`, `products` e
  `order_payments`.
- **O que será analisado:** evolução das vendas ao longo do tempo, categorias de produto mais
  relevantes, distribuição geográfica das vendas por estado, prazo de entrega e formas de
  pagamento utilizadas.
- **Fora do escopo desta etapa:** análise de avaliações/reviews (tabela `order_reviews`),
  geolocalização detalhada (lat/long) e qualquer modelo preditivo — reservados para etapas
  futuras, se solicitado.
- **Perguntas de negócio que guiam a análise:**
  1. As vendas têm sazonalidade ao longo do tempo?
  2. Quais categorias de produto geram mais receita?
  3. Em quais estados brasileiros a Olist mais vende?
  4. Como está o prazo de entrega aos clientes?
  5. Quais formas de pagamento os clientes mais utilizam?

*(Adapte os textos acima com as palavras do seu grupo.)*


## 3. Ambiente e bibliotecas

Notebook pensado para o **Google Colab**, usando **Pandas**, **NumPy** e **Matplotlib**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)


## 4. Baixando os dados (Kaggle)

O dataset está hospedado no Kaggle. Para baixar direto no Colab, é preciso de um **token de API
do Kaggle** (gratuito). Passo a passo:

1. Crie uma conta gratuita em [kaggle.com](https://www.kaggle.com) (se ainda não tiver).
2. Acesse **kaggle.com/settings** → seção **API** → botão **"Create New Token"**.
   Isso baixa um arquivo `kaggle.json` no seu computador.
3. Rode a célula abaixo e, quando o botão de upload aparecer, selecione o `kaggle.json` baixado.


In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o arquivo kaggle.json baixado do Kaggle


In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -o -q brazilian-ecommerce.zip -d olist_data
!ls olist_data


In [ ]:
caminho = 'olist_data/'

orders = pd.read_csv(caminho + 'olist_orders_dataset.csv',
                      parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                                   'order_estimated_delivery_date'])
customers = pd.read_csv(caminho + 'olist_customers_dataset.csv')
order_items = pd.read_csv(caminho + 'olist_order_items_dataset.csv')
products = pd.read_csv(caminho + 'olist_products_dataset.csv')
payments = pd.read_csv(caminho + 'olist_order_payments_dataset.csv')

print('orders:', orders.shape)
print('customers:', customers.shape)
print('order_items:', order_items.shape)
print('products:', products.shape)
print('payments:', payments.shape)


## 5. Análise exploratória com Pandas / NumPy

O dataset é relacional (várias tabelas), então a primeira etapa da análise exploratória é juntar
(`merge`) as tabelas necessárias, além de limpar e derivar colunas de interesse.


In [ ]:
orders.info()


In [ ]:
orders['order_status'].value_counts()


In [ ]:
# Mantemos apenas pedidos efetivamente entregues: representam vendas concretizadas
pedidos_entregues = orders[orders['order_status'] == 'delivered'].copy()
print(f"Pedidos entregues: {len(pedidos_entregues):,} de {len(orders):,} totais")


In [ ]:
# Junta itens do pedido + produto + pedido (só entregues) + cliente
itens = order_items.merge(products, on='product_id', how='left')

itens = itens.merge(
    pedidos_entregues[['order_id', 'customer_id', 'order_purchase_timestamp',
                        'order_delivered_customer_date', 'order_estimated_delivery_date']],
    on='order_id', how='inner'
)

itens = itens.merge(customers, on='customer_id', how='left')

print(itens.shape)
itens.head()


In [ ]:
# Colunas derivadas
itens['Receita'] = itens['price'] + itens['freight_value']
itens['AnoMes'] = itens['order_purchase_timestamp'].dt.to_period('M').astype(str)
itens['PrazoEntregaDias'] = (itens['order_delivered_customer_date'] - itens['order_purchase_timestamp']).dt.days

# Exemplo de uso do NumPy: sinaliza pedidos entregues fora do prazo estimado
itens['AtrasoEntrega'] = np.where(
    itens['order_delivered_customer_date'] > itens['order_estimated_delivery_date'],
    'Atrasado', 'No prazo'
)

itens[['Receita', 'AnoMes', 'PrazoEntregaDias', 'AtrasoEntrega']].describe(include='all').T


In [ ]:
# % de pedidos entregues com atraso (métrica de qualidade logística)
(itens['AtrasoEntrega'].value_counts(normalize=True) * 100).round(1)


## 6. Visualizações (Matplotlib)

Cinco visualizações a partir da fonte escolhida, cada uma com a justificativa de como apoia a
tomada de decisão.


### 6.1 Receita total por mês (sazonalidade)

In [ ]:
receita_mes = itens.groupby('AnoMes')['Receita'].sum().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(receita_mes.index, receita_mes.values, marker='o')
ax.set_title('Receita total por mês')
ax.set_xlabel('Mês'); ax.set_ylabel('Receita (R$)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Como ajuda na tomada de decisão:** revela picos e quedas de vendas ao longo do tempo (ex.:
datas comemorativas). Apoia o **planejamento de estoque** dos vendedores parceiros e o
**calendário de campanhas de marketing** da plataforma.

### 6.2 Top 10 categorias de produto por receita

In [ ]:
receita_cat = itens.groupby('product_category_name')['Receita'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(receita_cat.index[::-1], receita_cat.values[::-1])
ax.set_title('Top 10 categorias por receita')
ax.set_xlabel('Receita (R$)')
plt.tight_layout()
plt.show()


**Como ajuda na tomada de decisão:** aponta as **categorias mais lucrativas** da plataforma.
Apoia decisões de **recrutamento de novos vendedores** nessas categorias e de **destaque delas em
promoções e na página inicial**.

### 6.3 Top 10 estados por receita

In [ ]:
receita_estado = itens.groupby('customer_state')['Receita'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(receita_estado.index, receita_estado.values)
ax.set_title('Top 10 estados por receita')
ax.set_xlabel('Estado (UF)'); ax.set_ylabel('Receita (R$)')
plt.tight_layout()
plt.show()


**Como ajuda na tomada de decisão:** mostra a **concentração geográfica das vendas** no Brasil.
Apoia decisões de **expansão de centros de distribuição** e de **campanhas regionais de
marketing** focadas nos estados de maior potencial.

### 6.4 Distribuição do prazo de entrega

In [ ]:
prazo_valido = itens['PrazoEntregaDias'].dropna()
prazo_valido = prazo_valido[prazo_valido < prazo_valido.quantile(0.98)]  # remove outliers extremos

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(prazo_valido, bins=30)
ax.set_title('Distribuição do prazo de entrega (dias)')
ax.set_xlabel('Dias entre compra e entrega'); ax.set_ylabel('Número de pedidos')
plt.tight_layout()
plt.show()

print('Prazo médio de entrega (dias):', round(prazo_valido.mean(), 1))


**Como ajuda na tomada de decisão:** a entrega é um dos principais fatores de satisfação em
e-commerce. Esse gráfico apoia decisões de **negociação com transportadoras**, **escolha de
centros de distribuição mais próximos dos clientes** e ajuda a definir **prazos estimados mais
realistas** para o cliente no momento da compra.

### 6.5 Formas de pagamento utilizadas

In [ ]:
pedidos_pagamento = payments.merge(pedidos_entregues[['order_id']], on='order_id', how='inner')
contagem_pag = pedidos_pagamento['payment_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(contagem_pag.index, contagem_pag.values)
ax.set_title('Formas de pagamento utilizadas')
ax.set_ylabel('Número de pedidos')
plt.tight_layout()
plt.show()


**Como ajuda na tomada de decisão:** mostra a **preferência de pagamento** dos clientes
brasileiros (ex.: cartão de crédito x boleto). Apoia decisões sobre **quais meios de pagamento
priorizar no checkout**, negociação de taxas com adquirentes e estratégias de **parcelamento**.

## 7. Conclusão

*(Escreva aqui, com as palavras do grupo, um resumo dos principais achados e como, juntos, os
cinco gráficos ajudariam a gestão da Olist ou de um vendedor parceiro a tomar decisões de
estoque, logística, marketing e meios de pagamento. Vale citar limitações, como o fato de o
dataset cobrir apenas 2016–2018 e não incluir dados mais recentes.)*

## 8. Organização do repositório no GitHub

- [ ] Repositório criado com nome descritivo (ex.: `pi4-mineracao-dados-etapa1`)
- [ ] Todos os integrantes do grupo adicionados como colaboradores
- [ ] Este notebook (`.ipynb`) commitado no repositório
- [ ] `README.md` com o nome do grupo, a fonte de dados e como rodar o notebook
      (⚠️ não commitar o arquivo `kaggle.json` — ele é uma credencial pessoal)
- [ ] Link do notebook no Google Colab (opcional, facilita a correção) adicionado ao README
